## Running Ironhack notebooks in VS Code

### One-off setup (already done)

Your `.venv_airflow` environment already has `ipykernel` and `pandas` installed, so you don't need to repeat this. It's here in case you ever make a new environment or need a new package.

1. In the VS Code terminal, make sure you're in `~/data_course`.
2. Switch the environment on: `source .venv_airflow/bin/activate` (the prompt should start with `(.venv_airflow)`).
3. Install what notebooks need: `pip install ipykernel pandas`
4. For any other package a lab needs later (for example numpy), use the same pattern: `pip install <package name>`

### Every time you add a new notebook

Each new .ipynb file needs its kernel picking once. After that, VS Code remembers it for that file.

1. **Drag it in.** Drag the downloaded .ipynb from Windows into your `01_notebooks` folder in the VS Code Explorer (left-hand panel). Rename it if you like (right-click → Rename), for example replacing DATE with the date.
2. **Open it.** Click the file in the Explorer.
3. **Click Select Kernel.** It's in the top-right corner of the notebook. (If it already says `.venv_airflow`, skip to step 7.)
4. **Click Python Environments…**
5. **Pick .venv_airflow.** Choose the line ending `.venv_airflow/bin/python`. Ignore the one marked "Recommended" (`/usr/bin/python3`): that's Ubuntu's own Python and it won't work.
6. **Check the top-right.** It should now say `.venv_airflow`.
7. **Run the import cell.** Click ▷ next to the import cell. A green tick and [1] mean it's working.

In [2]:
from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path

## Part 1 — Task Boundaries, Portable Paths, and DAG Identity

### Q. 1 — Split One Large Task

A developer puts all of this work into one Airflow task:

```python
read source file
clean rows
build daily sales
validate output
```

Split it into separate tasks and show the dependency order.

========================================================================
ANSWER Q1
========================================================================

read_source_file >> clean_rows >> build_daily_sales >> validate_output

========================================================================

### Q. 2 — Build a DAG with Clear Task Boundaries

Create a complete DAG named:

```python
brightcart_task_boundaries
```

Tasks:

```python
read_source
clean_rows
build_daily_sales
validate_output
```

Use BashOperator and the correct dependency chain.

In [ ]:
# DAG in its own .py file in the lab folder.

### Q. 3 — Build a Portable File Path

Use Python's `pathlib`.

Create a base folder:

```python
brightcart_data
```

Then create these paths:

```python
raw/orders.csv
clean/orders_clean.csv
reports/daily_sales.csv
```

Do not hard-code Windows-only or Linux-only separators.

In [ ]:
from pathlib import Path

base_dir = (Path.home() / "data_course" / "26.11.12_BrightCart_Daily_Sales_Pipeline_Mini_Project" / "00_notebooks" / "26.11.10_Day_1_LAB_BrightCart_Airflow_Orchestration_and_Recovery" / "brightcart_data")
# Doing it this way instead of copying the path, prevents it being hard coded to one operators formatting. 
 
raw_file = (base_dir / "raw" / "orders.csv")
 
clean_file = (base_dir / "clean" / "orders_clean.csv")
 
report_file = (base_dir / "reports" / "daily_sales.csv")
 
print(raw_file)
print(clean_file)
print(report_file)

/home/dominicbrookes/data_course/airflow/dags/26.11.10_LAB_BrightCart_Airflow_Orchestration_and_Recovery/brightcart_data/raw/orders.csv
/home/dominicbrookes/data_course/airflow/dags/26.11.10_LAB_BrightCart_Airflow_Orchestration_and_Recovery/brightcart_data/clean/orders_clean.csv
/home/dominicbrookes/data_course/airflow/dags/26.11.10_LAB_BrightCart_Airflow_Orchestration_and_Recovery/brightcart_data/reports/daily_sales.csv


### Q. 4 — MCQ: Task Boundaries

Why split a pipeline into clear Airflow tasks?

A. To make every task use the same command

B. To remove all dependencies

C. To make failures and reruns easier to manage

D. To prevent logs from being created

========================================================================
ANSWER Q4
========================================================================

C. To make failures and reruns easier to manage.

========================================================================

## Part 2 — Start Date, Manual Runs, Scheduling, Catchup, and Backfill

### Q. 5 — Create a Manual-Run DAG

Create a complete DAG:

```python
dag_id = brightcart_manual_pipeline
schedule = None
catchup = False
```

Add one task:

```python
run_pipeline
```

In [ ]:
# DAG in its own .py file in the lab folder.

### Q. 6 — Create a Daily Scheduled DAG

Create a complete DAG:

```python
dag_id = brightcart_daily_pipeline
start_date = 2026-08-01
schedule = @daily
catchup = False
```

Add:

```python
extract
publish
```

with:

```python
extract -> publish
```

In [ ]:
# DAG in its own .py file in the lab folder.

### Q. 7 — Plan a Backfill Safely

The daily pipeline missed:

```python
2026-08-10
2026-08-11
2026-08-12
```

Write a safe recovery plan.

Include:

```python
check source data
check existing output
process dates in order
validate each date
check duplicates
```

========================================================================
ANSWER Q7
========================================================================

Ironhack's answer is eloquent, so will copy ironhacks answer. 

- Confirm source data exists for all three dates.

- Check whether any partial output already exists.

- Process 2026-08-10.

- Validate counts and totals.

- Process 2026-08-11.

- Validate counts and totals.

- Process 2026-08-12.

- Validate counts and totals.

- Confirm no duplicate output was created.

========================================================================

### Q. 8 — MCQ: Manual Run

Which DAG setting is suitable when you want no normal schedule and plan to trigger the DAG manually?

A. `schedule="@daily"`

B. `schedule=None`

C. `catchup=True`

D. `retries=0`

========================================================================
ANSWER Q8
========================================================================

B. schedule = None.

========================================================================

## Part 3 — Retries, Failure Propagation, Parse Errors, and Runtime Errors

### Q. 9 — Add a Controlled Retry

Create a complete DAG with:

```python
download_file
process_file
```

Requirements:

```python
download_file retries = 2
retry delay = 1 minute
download_file -> process_file
```

In [ ]:
# DAG in its own .py file in the lab folder.

### Q. 10 — Parse Error or Runtime Error?

Classify each problem.

```python
A. Missing closing bracket in the DAG Python file.
B. File path is valid Python but the file does not exist when the task runs.
C. Python import line has invalid syntax.
D. Database connection fails while a task is running.
```

========================================================================
ANSWER Q10
========================================================================

A. Missing closing bracket in the DAG Python file.

Parse error.

====================================

B. File path is valid Python but the file does not exist when the task runs.

Runtime error.

====================================

C. Python import line has invalid syntax.

Parse error.

====================================

D. Database connection fails while a task is running.

Runtime error.

========================================================================

### Q. 11 — Follow Failure Propagation

A DAG uses:

```python
extract -> clean -> report
```

States:

```python
extract = success
clean = failed
report = upstream_failed
```

Explain why `report` did not run normally.

========================================================================
ANSWER Q11
========================================================================

The report could not run while the clean failed; the dependency principle was broken. 

========================================================================

### Q. 12 — MCQ: Retry

Which problem is most suitable for automatic retry?

A. A Python syntax error

B. A missing required business column

C. An invalid DAG dependency definition

D. A temporary network timeout

========================================================================
ANSWER Q12
========================================================================

D. A temporary network timeout.

========================================================================

## Part 4 — Recovery Workflow and Output Freshness

### Q. 13 — Follow a Recovery Workflow

A task failed while creating:

```python
daily_sales.csv
```

Write the steps you should follow before simply rerunning it.

========================================================================
ANSWER Q13
========================================================================

You need to know what else is in the workflow. Assuming the failed task was the first task to fail in the dependency chain, then check the error message. If there are no clues in the error message, check whether the file arrived, whether there was a network timeout, etc. Did the task begin and not finish? Will rerunning it cause idempotency issues?

'A sensible recovery workflow is:

- Open the failed task log.

- Find the first useful error.

- Check whether the task created partial output.

- Fix the cause.

- Remove or safely replace partial output if needed.

- Rerun the failed work.

- Check downstream tasks.

- Validate the final output before considering the pipeline recovered.'

========================================================================

### Q. 14 — Check Output Freshness with Python

Use:

```python
from datetime import datetime, timedelta

expected_date = datetime(2026, 8, 15, 6, 0, 0)
output_modified = datetime(2026, 8, 15, 5, 55, 0)
```

Calculate the age of the output.

Treat it as fresh when it is no more than 15 minutes older than the expected time.

In [ ]:
from datetime import datetime, timedelta
 
expected_date = datetime(2026, 8, 15, 6, 0, 0)
output_modified = datetime(2026, 8, 15, 5, 55, 0)

def output_age(expected_date, output_modified):
    if expected_date - output_modified <= timedelta(minutes=15):
        print(f"Fresh. Age of output: {expected_date - output_modified}")
    else:
        print(f"Too old. Age of output: {expected_date - output_modified}")
    return(expected_date - output_modified)

age_of_output = output_age(expected_date, output_modified)
# In a real pipeline you’d usually want a True/False, because a later step (like stopping the DAG) can act on it, but it can’t act on a printed message.
# def output_age(expected_date, output_modified):
#     if expected_date - output_modified <= timedelta(minutes=15):
#         fresh = True
#         print(f"Fresh. Age of output: {expected_date - output_modified}")
#     else:
#         fresh = False
#         print(f"Too old. Age of output: {expected_date - output_modified}")
#     return(fresh, expected_date - output_modified)
# fresh, age_of_output = output_age(expected_date, output_modified)

# Ironhack's Answer:

expected_date = datetime(2026, 8, 15, 6, 0, 0)
 
output_modified = datetime(2026, 8, 15, 5, 55, 0)
 
age = expected_date - output_modified
is_fresh = (age <= timedelta(minutes=15))

print("Age:", age)
print("Fresh:", is_fresh)
    

### Q. 15 — Validate a Freshness Scenario

A report should be ready by:

```python
06:00
```

Observed output times:

```python
Batch 1 -> 05:55
Batch 2 -> 06:08
Batch 3 -> 06:40
```

The accepted delay is:

```python
15 minutes
```

Classify each output as acceptable or too late.

========================================================================
ANSWER Q15
========================================================================

Batch 1 -> 05:55

Acceptable.

====================================

Batch 2 -> 06:08

Acceptable.

====================================

Batch 3 -> 06:40

Too late.

========================================================================

### Q. 16 — MCQ: Output Freshness

What does output freshness tell you?

A. Whether the output was produced recently enough for its expected use

B. Whether the DAG file contains enough comments

C. Whether every source row is unique

D. Whether the scheduler has only one DAG

========================================================================
ANSWER Q16
========================================================================

A. Whether the output was produced recently enough for its expected use.

========================================================================

## Optional Practice

### Q. 17 — Build a Complete Recovery-Aware DAG

Create a complete DAG with:

```python
extract
clean
build_report
validate_output
```

Requirements:

```python
schedule = @daily
catchup = False
extract retries = 2
clean retries = 1
dependency chain:
extract -> clean -> build_report -> validate_output
```

In [ ]:
# DAG in its own .py file in the lab folder.

### Q. 18 — Fix a Non-Portable Path

A DAG contains:

```python
input_file = "C:\\brightcart\\raw\\orders.csv"
```

Rewrite the path using `pathlib` so the Python code is easier to move between operating systems.

In [ ]:
from pathlib import Path

base_dir = (Path.home() / "data_course" / "airflow" / "dags" / "26.11.10_LAB_BrightCart_Airflow_Orchestration_and_Recovery" / "brightcart_data")
# Really this would be whatever path I made for the exercise; I haven't made one though, so this is inaccurate. 
 
input_file = (base_dir / "raw" / "orders.csv")
 
print(input_file)


# Ironhack's Answer:

base_dir = Path("brightcart")

input_file = (base_dir / "raw" / "orders.csv")
 
print(input_file)

# 'Both answers are portable, because both build the path from separate pieces joined with /. The difference is where the path starts.
# Ironhack’s, Path("brightcart"), is a relative path. It means “a brightcart folder inside wherever the code is currently running from”. 
# It keeps the original structure (brightcart/raw/orders.csv) and simply drops the C:\ part. It’s short and fixes exactly what the question asked about, 
# the Windows-only backslashes and drive letter. The weakness is that the result depends on where you run it from, which is what you saw in Q3 when the notebook looked in 
# 01_notebooks. In Airflow it’s riskier still, because tasks often don’t run from the DAG’s folder.

# Yours, Path.home() / ..., is anchored to your home folder, so it always points to the same real folder whatever you run it from. Its weakness is that another machine needs 
# that exact data_course/airflow/dags/... structure under its home folder. It has also changed the location from the original brightcart folder to your lab’s brightcart_data folder, 
# which is fine for your setup but goes slightly beyond what the question asked.

# In a real DAG file, the common way to get the best of both is to anchor the path to the DAG file itself:

# base_dir = Path(__file__).parent / "brightcart_data"
# __file__ means “this .py file”, and .parent is the folder it’s in. So the path always points to brightcart_data next to the DAG, on any machine, wherever the folder lives. 
# This only works in a .py file, not in a notebook cell, which is why neither answer here uses it.'


### Q. 19 — Diagnose a Failed DAG

A scheduled run shows:

```python
extract = success
clean = failed
build_report = upstream_failed
validate_output = upstream_failed
```

The clean-task log shows:

```python
FileNotFoundError:
brightcart/raw/orders_2026_08_15.csv
```

Write the recovery steps.

========================================================================
ANSWER Q19
========================================================================

A sensible recovery is:

- Confirm the missing source file is expected for this run.

- Check whether the file arrived under a different name or path.

- Correct the path or restore the missing file.

- Confirm no partial clean output exists.

- Rerun or clear the failed clean task as appropriate.

- Allow build_report and validate_output to continue.

- Check the final report's row count, totals, and freshness.

========================================================================

### Q. 20 — Design a Daily Operational Check

BrightCart expects one successful daily pipeline run.

Create a checklist containing:

```python
DAG run status
failed task check
source availability
output existence
output freshness
row-count check
business-total check
```

Explain why checking only the DAG's green status is not enough.

========================================================================
ANSWER Q20
========================================================================

1. Confirm the expected DAG run exists.

2. Check its overall status.

3. Confirm no important task failed or was skipped unexpectedly.

4. Confirm the expected source was available.

5. Confirm the output exists.

6. Check that the output is fresh enough.

7. Compare expected and actual row counts.

8. Compare an important business total.


====================================

Explain why checking only the DAG's green status is not enough.

A green status only means the tasks ran without an error. The output could still be wrong, for example if a cleaning step was missing, or if the source only sent partial data. These checks need doing on every run, not just the first, because the source data changes each day.

Ironhack's answer: 'A green DAG status shows that Airflow completed the workflow according to its task rules. It does not by itself prove that the output contains the correct or latest business data.'

========================================================================